# TirraMind Phase 50 — Price Features + Residual Returns

**Zero config. Just attach the two datasets and click Run All.**

## Setup (one-time, already done if running via `kaggle_launch.py`)

- **Dataset 1:** `tirramind-data` → contains `pipeline.db`
- **Dataset 2:** `tirramind-code` → contains `agent/` + `scripts/` (auto-uploaded by `kaggle_launch.py`)
- **Accelerator:** GPU T4 x1 (Settings → Accelerator)
- W&B optional: add `WANDB_API_KEY` Kaggle Secret for live loss monitoring

In [ ]:
import hashlib, json

_NOTEBOOK_CONFIG = {
    "phase": "50l-stage1-ssl",
    "training_preset": "phase50_stage1_ssl",
    "resume_epoch": 0,
    "return_weight": 0.0,
    "obs_type_weight": 1.0,
    "time_delta_weight": 1.0,
    "value_weight": 1.0,
    "contrastive_weight": 1.0,
    "hidden_dim": 128,
    "num_layers": 2,
    "num_heads": 4,
    "epochs": 90,
    "window_size_h": 168,
    "gdelt_frac": 0.05,
    "defi_frac": 1.0,
    "n1_doctrine": False,
    "max_windows": 200,
    "vicreg_weight": 0.0,
    "use_contranorm": False,
    "use_log_loss": False,
    "eval_smoke": False,
    "run_full_backtest": False,
    "skip_eval": False,
    "skip_retrain_split_eval": True,
    "post_train_eval": True,
    "run_stage2_eval": True,
    "auto_tune": False,
    "listnet": False,
    "direction_loss": False,
    "residual_returns": False,
    "use_csrc_loss": False,
    "use_concat_head": False,
    "use_concat_batchnorm": False,
    "use_pcgrad": False,
    "primary_ic_strategy": "GNN-PurgedRanker",
    "time_delta_nan_fix": True,
    "xsnorm_price_feats": True,
    "obs_type_ce_clamp": 20.0,
    "kernel_version": 73,
    "fix": "v73_stage1_ssl"
}

_cfg_str  = json.dumps(_NOTEBOOK_CONFIG, sort_keys=True, separators=(",", ":"))
_fp_full  = hashlib.sha256(_cfg_str.encode()).hexdigest()
_fp_short = _fp_full[:12]

_W = 60
_banner = [
    "╔" + "═" * _W + "╗",
    f"║{f'  TirraMind — Phase 50l  ·  Kernel v{_NOTEBOOK_CONFIG["kernel_version"]}':^{_W}}║",
    f"║{'Phase B Stage-1 SSL  |  no return/CSRC/concat head':^{_W}}║",
    "╠" + "═" * _W + "╣",
    f"║{'':^{_W}}║",
    f"║  {'use_concat_head':<26} {'✓ GNN embeddings → return prediction':<{_W-30}}  ║",
    f"║  {'csrc_loss':<26} {'✓ return-decile-based pairs':<{_W-30}}  ║",
    f"║  {'freeze_backbone':<26} {'✗ full GNN training':<{_W-30}}  ║",
    f"║  {'resume_epoch':<26} {_NOTEBOOK_CONFIG['resume_epoch']!s:<{_W-30}}  ║",
    f"║  {'epochs (target)':<26} {_NOTEBOOK_CONFIG['epochs']!s:<{_W-30}}  ║",
    f"║{'':^{_W}}║",
    "╠" + "═" * _W + "╣",
    f"║  CONFIG FINGERPRINT  :  {_fp_short:<{_W-27}}║",
    "╚" + "═" * _W + "╝",
]
print("\n".join(_banner))
print(f"\n✓ Fingerprint: {_fp_short}")










In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

# ── GPU compatibility fix (MUST run before any `import torch`) ────────────────
# PyTorch 2.6+ dropped P100 (sm_60). Kaggle assigns T4 or P100 randomly.
# If P100 is detected via nvidia-smi, reinstall torch 2.5.1 (last sm_60-compatible
# version) BEFORE torch is imported — so the kernel loads the right version.
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    gpu_line = r.stdout.strip().split("\n")[0] if r.stdout.strip() else ""
    print(f"nvidia-smi: {gpu_line}")

    if gpu_line:
        cap = float(gpu_line.split(",")[-1].strip())  # e.g. "6.0"
        if cap < 7.0:
            print(f"⚠ GPU sm_{cap} < 7.0 — reinstalling PyTorch 2.5.1 for P100 compatibility (~8 min)")
            pip(
                "--force-reinstall",
                "torch==2.5.1",
                "torchvision==0.20.1",
                "--index-url", "https://download.pytorch.org/whl/cu121",
            )
            print("✓ PyTorch 2.5.1 installed — P100 now usable as GPU")
        else:
            print(f"✓ GPU sm_{cap} ≥ 7.0 — PyTorch 2.10 OK (T4/A100)")
except Exception as _e:
    print(f"nvidia-smi check failed ({_e}) — proceeding with default PyTorch")

# ── Install PyG and helpers ───────────────────────────────────────────────────
# torch-geometric 2.7+ uses native PyTorch sparse ops — no pyg wheels needed.
pip("torch-geometric==2.7.0")
pip("tqdm", "rich", "wandb")

# After P100 torch reinstall, numpy is often 2.x but base scipy may be stale.
# NEVER downgrade numpy on Kaggle (breaks jax/cupy/etc and causes numpy.rec errors).
# Upgrade scipy only so import scipy.stats works in the IC eval cell.
pip("--upgrade", "scipy>=1.14.0")

# ── Verify ───────────────────────────────────────────────────────────────────
import numpy as np
import torch
import torch_geometric
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    print(f"Compute cap  : sm_{cap[0]}{cap[1]}")
print(f"PyG          : {torch_geometric.__version__}")
print(f"NumPy        : {np.__version__}")
import scipy
print(f"SciPy        : {scipy.__version__}")
print("Install OK.")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── CODE: try git clone with token, fall back to tirramind-code dataset ──────
_code_loaded = False

try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("tirramind_token")
    _repo_url = f"https://{_token}@github.com/savabs/tirramind.git"
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    subprocess.run(["git", "clone", "--depth=1", _repo_url, str(WORK_DIR)],
                   check=True, capture_output=True)
    print("✓ Code: cloned from GitHub (tirramind_token secret)")
    _code_loaded = True
except Exception as _e:
    print(f"  Git clone skipped ({_e.__class__.__name__}) — using tirramind-code dataset")

if not _code_loaded:
    def find_code_root(root="/kaggle/input"):
        for dirpath, dirs, _ in os.walk(root):
            if {"agent", "scripts"}.issubset(set(dirs)):
                return Path(dirpath)
        return None
    _code_root = find_code_root()
    assert _code_root is not None, (
        "CODE NOT FOUND.\n"
        "Fix: run  python scripts/kaggle_launch.py  locally — it uploads tirramind-code automatically.\n"
        "Or add tirramind_token Kaggle Secret for GitHub clone."
    )
    for _name in ("agent", "scripts"):
        _dst = WORK_DIR / _name
        if _dst.exists():
            shutil.rmtree(_dst)
        shutil.copytree(_code_root / _name, _dst)
    print(f"✓ Code: loaded from dataset at {_code_root}")

# ── DATA: pipeline.db from tirramind-data dataset ────────────────────────────
def find_data_root(root="/kaggle/input"):
    for dirpath, dirs, files in os.walk(root):
        if "pipeline.db" in set(files):
            return Path(dirpath)
    return None

_data_root = find_data_root()
assert _data_root is not None, (
    "DATA NOT FOUND.\n"
    "Fix: attach 'tirramind-data' dataset in the right panel → Data → Add Dataset."
)
pipeline_dir = WORK_DIR / ".tirra_pipeline"
pipeline_dir.mkdir(exist_ok=True)
shutil.copy2(_data_root / "pipeline.db", pipeline_dir / "pipeline.db")
print(f"✓ Data: pipeline.db ({(pipeline_dir / 'pipeline.db').stat().st_size // 1_000_000} MB)")

# ── Patch pipeline __init__ for lazy APScheduler import ──────────────────────
(WORK_DIR / "agent" / "pipeline" / "__init__.py").write_text(
    '"""TirraMind — Pipeline Layer."""\n\n'
    'from agent.pipeline.storage_backend import PostgresBackend, SQLiteBackend, StorageBackend\n'
    'from agent.pipeline.store import PipelineStore\n\n'
    '__all__ = ["PipelineStore", "PipelineScheduler", "StorageBackend", "SQLiteBackend", "PostgresBackend"]\n\n'
    'def __getattr__(name):\n'
    '    if name == "PipelineScheduler":\n'
    '        from agent.pipeline.scheduler import PipelineScheduler\n'
    '        return PipelineScheduler\n'
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n',
    encoding="utf-8",
)
print("✓ Patched pipeline __init__")
print("\nSetup complete — ready to train.")

In [ ]:
import torch
import torch_geometric

# ── Device selection ──────────────────────────────────────────────────────────
# Cell 1 already reinstalled PyTorch 2.5.1 if P100 was detected,
# so both T4 (sm_75) and P100 (sm_60) are now fully GPU-compatible.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_WINDOWS = 200  # full data — both T4 and P100 now supported

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU          : {gpu_name}  (sm_{cap[0]}{cap[1]})")
    print(f"Device       : {DEVICE}  |  max_windows: {MAX_WINDOWS}")
else:
    print("GPU          : none (CPU mode)")
    DEVICE = "cpu"

print(f"PyTorch      : {torch.__version__}")
print(f"PyG          : {torch_geometric.__version__}")

In [ ]:
from pathlib import Path
import shutil
import os
import re

WORK_DIR = Path("/kaggle/working/tirramind_v1")
assert (WORK_DIR / ".tirra_pipeline/pipeline.db").exists()
for f in ["agent/models/gnn/graph_builder.py", "agent/models/gnn/het_tgn.py",
          "agent/models/gnn/trainer.py", "scripts/retrain_gnn.py"]:
    assert (WORK_DIR / f).exists(), f"Missing: {f}"

CKPT_DIR = Path("/kaggle/working/phase50_ckpts")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

_resume_epoch = 0
_src_ckpt: Path | None = None

# ── Auto-discover highest available epoch from all input datasets ─────────────
# Walks /kaggle/input, collects all epoch_NNN.pt files, picks the highest N.
# If resume_epoch in _NOTEBOOK_CONFIG is 0, we skip discovery to train from scratch.
_config_resume = globals().get("_NOTEBOOK_CONFIG", {}).get("resume_epoch", 0)

if _config_resume > 0:
    _all_ckpts: list[tuple[int, Path]] = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        for fname in files:
            m = re.match(r"epoch_(\d+)\.pt$", fname)
            if m:
                _all_ckpts.append((int(m.group(1)), Path(root) / fname))

    if _all_ckpts:
        _all_ckpts.sort(key=lambda x: x[0], reverse=True)
        _best_epoch, _src_ckpt = _all_ckpts[0]
        print(f"Available checkpoints: {[f'ep{e}' for e, _ in _all_ckpts]}")
        shutil.copy2(_src_ckpt, CKPT_DIR / _src_ckpt.name)
        _resume_epoch = _best_epoch
        print(f"✓ Selected: {_src_ckpt.name}  ({_src_ckpt.stat().st_size / 1_000_000:.1f} MB) → {CKPT_DIR}")
    else:
        print("⚠ No epoch_NNN.pt found in any input dataset — training from epoch 1.")
else:
    print("✓ Configured to train from scratch (resume_epoch=0).")

print(f"All checks passed. Resuming from epoch {_resume_epoch}.")

In [ ]:
import subprocess, sys, os
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
CKPT_DIR = Path("/kaggle/working/phase50_ckpts")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = globals().get("DEVICE", "cpu")
_resume_epoch = globals().get("_resume_epoch", 0)
cfg = globals().get("_NOTEBOOK_CONFIG", {})

_wandb_project = None
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = _wandb_key
    _wandb_project = "tirramind"
    print("W&B enabled")
except Exception as _e:
    print(f"W&B disabled ({_e.__class__.__name__})")

print(f"Device: {DEVICE}")
_head = 'concat' if cfg.get('use_concat_head') else ('n1' if cfg.get('n1_doctrine') else 'pred')
print(f"Resuming from epoch {_resume_epoch} → epoch {cfg.get('epochs', 2)} (V{cfg.get('kernel_version', '?')} head={_head}, vicreg={cfg.get('vicreg_weight', 0)})")

cmd = [
    sys.executable, "scripts/retrain_gnn.py",
    "--epochs",              "10",
    "--hidden-dim",          str(cfg.get("hidden_dim", 128)),
    "--num-layers",          str(cfg.get("num_layers", 2)),
    "--num-heads",           str(cfg.get("num_heads", 4)),
    "--lr",                  "1e-3",
    "--backup",
    "--window-size",         "604800",
    "--gdelt-frac",          str(cfg.get("gdelt_frac", 1.0)),
    "--defi-frac",           str(cfg.get("defi_frac", 1.0)),
    "--max-windows",         str(cfg.get("max_windows", 0)),
    "--return-weight",       str(cfg.get("return_weight", 5.0)),
    "--obs-type-weight",     str(cfg.get("obs_type_weight", 0.0)),
    "--time-delta-weight",   str(cfg.get("time_delta_weight", 0.0)),
    "--value-weight",        str(cfg.get("value_weight", 0.0)),
    "--contrastive-weight",  str(cfg.get("contrastive_weight", 1.0)),
]

if cfg.get("listnet", False):
    cmd += ["--listnet", "--listnet-temperature", str(cfg.get("listnet_temperature", 0.1))]
if cfg.get("direction_loss", False):
    cmd += ["--direction-loss"]
if cfg.get("residual_returns", False):
    cmd += ["--residual-returns"]


if cfg.get("use_csrc_loss", True):
    cmd += [
        "--csrc-loss",
        "--csrc-temperature",    str(cfg.get("csrc_temperature", 0.1)),
        "--csrc-n-deciles",      str(cfg.get("csrc_n_deciles", 5)),
    ]

cmd += [
    "--device",              DEVICE,
    "--checkpoint-dir",      str(CKPT_DIR),
    "--model-out",           ".tirra_pipeline/gnn_model_phase50.pt",
    "--max-ram-gb",          "26.0",
]

_vicreg = float(cfg.get("vicreg_weight", 0) or 0)
if _vicreg > 0:
    cmd += ["--vicreg-weight", str(_vicreg)]

if cfg.get("use_contranorm", False):
    cmd += ["--use-contranorm"]
if cfg.get("use_log_loss", False):
    cmd += ["--use-log-loss"]

if cfg.get("n1_doctrine", False):
    cmd += ["--n1-doctrine"]
elif cfg.get("use_concat_head", False):
    cmd += ["--use-concat-head"]

_return_clamp = cfg.get("return_pred_clamp")
if _return_clamp is not None:
    cmd += ["--return-pred-clamp", str(_return_clamp)]
if cfg.get("use_concat_batchnorm", False):
    cmd += ["--use-concat-batchnorm"]
if cfg.get("use_pcgrad", False):
    cmd += ["--use-pcgrad"]

if cfg.get("auto_tune", False):
    cmd += ["--auto-tune"]
    cmd += ["--return-log-var-max", str(cfg.get("return_log_var_max", "-0.693"))]
    cmd += ["--contrastive-log-var-min", str(cfg.get("contrastive_log_var_min", "0.0"))]

if _resume_epoch > 0:
    cmd += ["--resume", str(_resume_epoch)]

if cfg.get("skip_eval", False) or cfg.get("skip_retrain_split_eval", False) or cfg.get("post_train_eval", False):
    cmd += ["--skip-eval"]

if _wandb_project:
    _start = _resume_epoch + 1 if _resume_epoch else 1
    cmd += [
        "--wandb-project", _wandb_project,
        "--wandb-run",     f"phase50-v{cfg.get('kernel_version', 63)}-ep{_start}-{cfg.get('epochs', 10)}-v52-smoke-gate",
        "--wandb-tags",    f"phase50,v{cfg.get('kernel_version', 63)},v52-exact,200win,ep3-gate,smoke",
    ]

print("Running:", " ".join(cmd))
print("-" * 70)

process = subprocess.Popen(
    cmd, cwd=str(WORK_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed: exit {return_code}")
print("\nPhase 50m training completed.")

In [ ]:
_run_post = cfg.get("post_train_eval", False) and not cfg.get("skip_eval", False)
if not _run_post:
    print("post_train_eval=False — skipping post-train eval (use tirramind-phase50-eval kernel).")
else:
    import json
    import subprocess
    import sys
    from pathlib import Path

    WORK_DIR = Path("/kaggle/working/tirramind_v1")
    CKPT_DIR = Path("/kaggle/working/phase50_ckpts")
    ckpts = sorted(CKPT_DIR.glob("epoch_*.pt"))
    if not ckpts:
        raise RuntimeError("No checkpoints in phase50_ckpts")
    last_ckpt = ckpts[-1]
    model_path = WORK_DIR / ".tirra_pipeline/gnn_model_phase50.pt"
    smoke = cfg.get("eval_smoke", int(cfg.get("epochs", 90)) <= 5)

    print(f"Post-train eval: {last_ckpt.name}  smoke={smoke}")

    diag_cmd = [
        sys.executable, "scripts/gnn_eval_diagnostics.py",
        "--model-path", str(model_path),
        "--weights-from-epoch", str(last_ckpt),
        "--db-path", ".tirra_pipeline/pipeline.db",
        "--checkpoint-dir", str(CKPT_DIR),
        "--notebook-config", json.dumps(cfg),
        "--out", "/kaggle/working/eval_diagnostics.json",
    ]
    if smoke:
        diag_cmd.append("--smoke")
    print("Running:", " ".join(diag_cmd))
    subprocess.run(diag_cmd, check=True, cwd=str(WORK_DIR))

    ic_out = Path("/kaggle/working/ic_results_phase50.json")
    bt_cmd = [
        sys.executable, "scripts/phase40_gnn_backtest.py",
        "--model-path", str(model_path),
        "--weights-from-epoch", str(last_ckpt),
        "--db-path", ".tirra_pipeline/pipeline.db",
        "--out", str(ic_out),
    ]
    if smoke:
        bt_cmd.append("--smoke")
    bt_cmd += [
        "--primary-ic-strategy",
        str(cfg.get("primary_ic_strategy", "GNN-ConcatReturnHead")),
    ]
    print("Running:", " ".join(bt_cmd))
    subprocess.run(bt_cmd, check=True, cwd=str(WORK_DIR))
    print("Post-train phase40 complete →", ic_out)

    if cfg.get("run_stage2_eval") or cfg.get("training_preset"):
        s2_out = Path("/kaggle/working/stage2_ranker_eval.json")
        s2_cmd = [
            sys.executable, "scripts/stage2_ranker_eval.py",
            "--checkpoint", str(model_path),
            "--weights-from-epoch", str(last_ckpt),
            "--db-path", ".tirra_pipeline/pipeline.db",
            "--out", str(s2_out),
        ]
        if smoke:
            s2_cmd.append("--smoke")
        print("Running:", " ".join(s2_cmd))
        subprocess.run(s2_cmd, check=True, cwd=str(WORK_DIR))
        print("Post-train Stage2 complete →", s2_out)


In [ ]:
# Optional full walk-forward IC (40 folds, ~45 min GPU). Off by default for smokes.
import subprocess, sys, shutil
from pathlib import Path

cfg = globals().get("_NOTEBOOK_CONFIG", {})
if not cfg.get("run_full_backtest", False):
    print("run_full_backtest=False — skipping full 40-fold backtest (post-train smoke eval already ran).")
else:
    WORK_DIR = Path("/kaggle/working/tirramind_v1")
    MODEL = WORK_DIR / ".tirra_pipeline" / "gnn_model_phase50.pt"
    assert MODEL.exists(), "Model not found — training may not have completed"

    # Symlink to expected name
    link = WORK_DIR / ".tirra_pipeline" / "gnn_model.pt"
    if link.exists():
        link.unlink()
    shutil.copy2(MODEL, link)

    result = subprocess.run(
        [sys.executable, str(WORK_DIR / "scripts/phase40_gnn_backtest.py"), "--out", ".tirra_pipeline/ic_results_phase50.json"],
        cwd=str(WORK_DIR), text=True
    )

    if result.returncode == 0:
        print("\nBacktest complete. Check output above for IC results.")
    else:
        print("Backtest FAILED")